# Object Detection for Autonomous Vehicles

You're an ML engineer at an autonomous vehicle company. Your team has heard about a pretrained object detection model on HuggingFace and wants to know whether it's any good for your use case: detecting pedestrians, vehicles, and cyclists from dashcam footage.

Your job is to pull the model down, run it against some dashcam images, and assess whether it's fit for purpose.

## What is object detection?

Object detection is the task of finding objects in images. Specifically, a model will:

- Draw **bounding boxes** around objects it finds
- Assign a **label** to each box (e.g. "person", "car", "bicycle")
- Give each detection a **confidence score** between 0 and 1, indicating how certain it is

## What is a pretrained model?

A pretrained model is one that someone else has already trained on a large dataset. We download it and use it directly — no training required on our part. This is extremely common in practice: training a model from scratch takes enormous amounts of data, compute, and expertise.

## Why evaluate, not build?

We're not building a model today. We're evaluating an existing one. The question is: does this model work well enough for our use case? That's a judgement call, and it's one of the most important skills in applied ML.

## Loading the model

In [ ]:
import micropip
import base64
from pathlib import Path

The next cell downloads a real neural network into your browser. This is YOLOS-tiny, a lightweight object detection model (~25MB). It may take 10–30 seconds to download — that's normal.

In [ ]:
await micropip.install("transformers_js_py")
from transformers_js_py import import_transformers_js
transformers = await import_transformers_js()
detector = await transformers.pipeline("object-detection", "Xenova/yolos-tiny")
print("Model loaded!")

## Running detection on dashcam images

Now we'll loop through all the JPEG images in the `images/` directory, run the detector on each one, and collect the results. Each result is a list of detections — each detection has a label, a confidence score, and a bounding box.

In [ ]:
from PIL import Image as PILImage
import matplotlib.pyplot as plt

image_dir = Path("./images")
image_paths = sorted(image_dir.glob("*.jpg"))


def image_to_data_url(path):
    """Convert a local image to a base64 data URL that transformers_js_py can read."""
    data = Path(path).read_bytes()
    b64 = base64.b64encode(data).decode("utf-8")
    return f"data:image/jpeg;base64,{b64}"


all_results = {}
for path in image_paths:
    result = await detector(image_to_data_url(path))
    all_results[path.name] = result
    print(f"{path.name}: {len(result)} detections")

Let's look at the raw output for one image to understand the structure. Each detection is a dictionary with `label`, `score`, and `box` (containing `xmin`, `ymin`, `xmax`, `ymax`).

In [ ]:
# Show detailed results for the first image
if all_results:
    first_image = list(all_results.keys())[0]
    print(f"Detections for {first_image}:\n")
    for det in all_results[first_image]:
        print(f"  {det['label']:>15s}  confidence: {det['score']:.2%}  box: {det['box']}")

## Visualising detections

Numbers are useful, but for object detection you really need to see the boxes on the images. The function below draws bounding boxes and labels on an image using matplotlib. It filters out any detections below a given confidence threshold.

In [ ]:
import matplotlib.patches as patches


def visualise_detections(image_path, detections, threshold=0.5):
    img = PILImage.open(image_path)
    fig, ax = plt.subplots(1, figsize=(10, 7))
    ax.imshow(img)

    for det in detections:
        if det["score"] < threshold:
            continue
        box = det["box"]
        x, y = box["xmin"], box["ymin"]
        w, h = box["xmax"] - box["xmin"], box["ymax"] - box["ymin"]
        rect = patches.Rectangle(
            (x, y), w, h, linewidth=2, edgecolor="lime", facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(
            x,
            y - 5,
            f'{det["label"]} ({det["score"]:.0%})',
            color="lime",
            fontsize=10,
            fontweight="bold",
            backgroundcolor="black",
        )

    ax.set_title(Path(image_path).name)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

Let's visualise a few of the images with their detections.

In [ ]:
for path in image_paths[:3]:
    visualise_detections(path, all_results[path.name])

## Evaluating quality

### Which objects does the model detect reliably?

Let's count up all the detections by label across every image. This gives us a rough picture of what the model is finding.

In [ ]:
from collections import Counter

label_counts = Counter()
for image_name, detections in all_results.items():
    for det in detections:
        label_counts[det["label"]] += 1

print("Detection counts by label (all images, no threshold):\n")
for label, count in label_counts.most_common():
    print(f"  {label:>15s}: {count}")

### Which objects does it miss or misclassify?

Look back at the visualisations above. Think about:

- Are there objects you can see in the image that the model didn't detect?
- Are there any detections that are clearly wrong (a person labelled as a car, for example)?
- Are the bounding boxes tight around the objects, or are they sloppy?

### What confidence threshold should we use?

The model assigns a confidence score to every detection. A higher threshold means we only keep detections the model is very sure about — fewer false positives, but more missed objects. A lower threshold catches more objects but lets in more noise.

Run the cell below to see the same image at three different thresholds. Watch how the detections change.

In [ ]:
if image_paths:
    example_path = image_paths[0]
    for threshold in [0.3, 0.5, 0.98]:
        print(f"\n--- Threshold: {threshold} ---")
        visualise_detections(example_path, all_results[example_path.name], threshold=threshold)

### Are there failure patterns?

Models tend to fail in predictable ways. Think about whether any of these apply:

- **Small objects:** Are distant pedestrians or cyclists being missed?
- **Occlusion:** Does the model struggle when objects are partially hidden behind other objects?
- **Lighting:** How does it handle glare, shadow, dusk, or night-time images?
- **Unusual angles:** Dashcam footage has a specific perspective — does the model cope with it?

## Fitness for purpose

Now step back from the technical details and think about the bigger picture.

### 1. Would you trust this model to detect pedestrians at 60mph? Why or why not?

### 2. What is the gap between "works on these test images" and "safe for deployment"?

### 3. What additional testing would you want before recommending adoption to the safety board?

## Model provenance

Before you recommend any model, you need to know where it came from. A **model card** is a document that accompanies a model, describing what it is, how it was trained, and what its limitations are.

Let's read the model card for YOLOS-tiny.

In [ ]:
with open("./model-card.md") as f:
    print(f.read())

Now consider the following questions.

**Who trained this model? On what data?**

**What are the stated limitations?**

**What's NOT in the model card that you'd want to know?**

**Would you trust a model if the card was missing?**